# Fase 3 — Algoritmos, recursividad y complejidad

## Propósito del notebook

Este notebook corresponde al avance de la Formativa 3 del proyecto “Optimización del OEE mediante Mantenimiento Predictivo Inteligente en Activos Críticos Industriales”.

En continuidad con la Fase 2, se reutiliza el dataset procesado ubicado en `data/processed/dataset_procesado.csv`. Esta fase no reconstruye el pipeline de limpieza, sino que prepara la base para implementar algoritmos estructurados, algoritmos recursivos y mediciones de complejidad temporal y/o espacial.

In [37]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

In [38]:
def obtener_raiz_proyecto() -> Path:
    """
    Identifica la raíz del proyecto según la ubicación desde donde se ejecuta el notebook.
    Permite ejecutar el notebook desde la carpeta F3 o desde la raíz del repositorio.
    """
    cwd = Path.cwd()

    if (cwd / "data").exists():
        return cwd

    if (cwd.parent / "data").exists():
        return cwd.parent

    raise FileNotFoundError("No se pudo identificar la raíz del proyecto.")


PROJECT_ROOT = obtener_raiz_proyecto()
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed" / "dataset_procesado.csv"

print("Raíz del proyecto:", PROJECT_ROOT)
print("Dataset procesado:", DATA_PROCESSED)

Raíz del proyecto: c:\Python\mcdi500_s1_grupo7
Dataset procesado: c:\Python\mcdi500_s1_grupo7\data\processed\dataset_procesado.csv


In [39]:
def cargar_dataset_procesado(ruta: Path) -> pd.DataFrame:
    """
    Carga el dataset procesado de Fase 2 y convierte la columna date a formato datetime.
    
    Parámetros:
        ruta (Path): ruta relativa o absoluta del archivo CSV procesado.
    
    Retorna:
        pd.DataFrame: dataset cargado y preparado para validación inicial de Fase 3.
    """
    if not ruta.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {ruta}")

    df = pd.read_csv(ruta, parse_dates=["date"])

    print(f"Dataset cargado correctamente: {df.shape[0]} filas y {df.shape[1]} columnas.")
    return df


df = cargar_dataset_procesado(DATA_PROCESSED)
df.head()

Dataset cargado correctamente: 124493 filas y 15 columnas.


,date,device,failure,metric1,metric2,metric3,metric4,metric5,metric6,metric7,metric8,metric9,year,month,day
0,2015-01-01,S1F01085,0,215630672,55,0,52,6,407438,0,0,7,2015,1,1
1,2015-01-01,S1F0166B,0,61370680,0,3,0,6,403174,0,0,0,2015,1,1
2,2015-01-01,S1F01E6Y,0,173295968,0,0,0,12,237394,0,0,0,2015,1,1
3,2015-01-01,S1F01JE0,0,79694024,0,0,0,6,410186,0,0,0,2015,1,1
4,2015-01-01,S1F01R2B,0,135970480,0,0,0,15,313173,0,0,3,2015,1,1


In [40]:
def validar_dataset_f3(df: pd.DataFrame) -> None:
    """
    Valida condiciones mínimas del dataset procesado para continuar con la Fase 3.
    No reemplaza el pipeline de Fase 2; solo confirma que el insumo está listo para análisis algorítmico.
    """
    columnas_esperadas = (
        ["date", "device", "failure"] +
        [f"metric{i}" for i in range(1, 10)] +
        ["year", "month", "day"]
    )

    columnas_faltantes = [col for col in columnas_esperadas if col not in df.columns]

    assert len(columnas_faltantes) == 0, f"Faltan columnas esperadas: {columnas_faltantes}"
    assert df.shape[0] == 124493, f"Cantidad de filas inesperada: {df.shape[0]}"
    assert df.isnull().sum().sum() == 0, "Existen valores nulos en el dataset."
    assert df.duplicated().sum() == 0, "Existen registros duplicados en el dataset."
    assert pd.api.types.is_datetime64_any_dtype(df["date"]), "La columna date no está en formato datetime."
    assert set(df["failure"].unique()).issubset({0, 1}), "La variable failure contiene valores distintos de 0 y 1."

    print("Validaciones automáticas F3: OK")


validar_dataset_f3(df)

Validaciones automáticas F3: OK


In [41]:
def resumen_dataset_f3(df: pd.DataFrame) -> pd.DataFrame:
    """
    Genera un resumen técnico del dataset utilizado como insumo para Fase 3.
    """
    resumen = pd.DataFrame({
        "criterio": [
            "Filas",
            "Columnas",
            "Valores nulos",
            "Duplicados",
            "Fecha mínima",
            "Fecha máxima",
            "Dispositivos únicos",
            "Eventos de falla",
            "Registros sin falla"
        ],
        "resultado": [
            df.shape[0],
            df.shape[1],
            int(df.isnull().sum().sum()),
            int(df.duplicated().sum()),
            df["date"].min(),
            df["date"].max(),
            df["device"].nunique(),
            int(df["failure"].sum()),
            int((df["failure"] == 0).sum())
        ]
    })

    return resumen


resumen_f3 = resumen_dataset_f3(df)
resumen_f3

,criterio,resultado
0,Filas,124493
1,Columnas,15
2,Valores nulos,0
3,Duplicados,0
4,Fecha mínima,2015-01-01 00:00:00
5,Fecha máxima,2015-11-02 00:00:00
6,Dispositivos únicos,1169
7,Eventos de falla,106
8,Registros sin falla,124387


## Validación inicial del dataset para Fase 3

El dataset utilizado en esta fase corresponde al archivo procesado en Fase 2. No se reconstruye el pipeline de limpieza, ya que el objetivo de la Formativa 3 es avanzar hacia el diseño de algoritmos, recursividad y mediciones de complejidad.

La validación inicial confirma que el dataset mantiene las condiciones esperadas: ausencia de valores nulos, ausencia de duplicados, columna `date` en formato temporal, presencia de variables derivadas `year`, `month` y `day`, y variable objetivo binaria `failure`.

Estas condiciones permiten continuar con las actividades de los otros integrantes del equipo: implementación de algoritmos estructurados, comparación de eficiencia y desarrollo de un algoritmo recursivo.

In [42]:
SRC_PATH = PROJECT_ROOT / "F3" / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

#from carga_validacion import cargar_dataset_procesado, validar_dataset_f3

#df_prueba = cargar_dataset_procesado(DATA_PROCESSED)
#validar_dataset_f3(df_prueba)

print("Importación desde script F3: OK")

Importación desde script F3: OK


In [43]:
import pandas as pd
from pathlib import Path

# Ruta directa al dataset procesado
ruta = Path.cwd().parent / "data" / "processed" / "dataset_procesado.csv"

df_prueba = pd.read_csv(ruta)

print("Dataset cargado correctamente ✅")
df_prueba.head()


Dataset cargado correctamente ✅


,date,device,failure,metric1,metric2,metric3,metric4,metric5,metric6,metric7,metric8,metric9,year,month,day
0,2015-01-01,S1F01085,0,215630672,55,0,52,6,407438,0,0,7,2015,1,1
1,2015-01-01,S1F0166B,0,61370680,0,3,0,6,403174,0,0,0,2015,1,1
2,2015-01-01,S1F01E6Y,0,173295968,0,0,0,12,237394,0,0,0,2015,1,1
3,2015-01-01,S1F01JE0,0,79694024,0,0,0,6,410186,0,0,0,2015,1,1
4,2015-01-01,S1F01R2B,0,135970480,0,0,0,15,313173,0,0,3,2015,1,1


## Algoritmo Recursivo y Análisis de Complejidad

In [44]:
# Seleccionar columna metric1 y crear muestra de 5000 datos
datos_metric1 = df_prueba["metric1"].head(5000).tolist()

print("Cantidad de elementos en la muestra:", len(datos_metric1))

Cantidad de elementos en la muestra: 5000


In [45]:
# Función merge: combina dos listas ordenadas
def merge(izquierda, derecha):
    resultado = []
    i = j = 0
    
    # Comparar elementos de ambas listas
    while i < len(izquierda) and j < len(derecha):
        if izquierda[i] < derecha[j]:
            resultado.append(izquierda[i])
            i += 1
        else:
            resultado.append(derecha[j])
            j += 1
    
    # Agregar elementos restantes
    resultado.extend(izquierda[i:])
    resultado.extend(derecha[j:])
    
    return resultado

In [46]:
# Función recursiva Merge Sort
def merge_sort(lista):
    
    # Caso base: lista de 1 o 0 elementos ya está ordenada
    if len(lista) <= 1:
        return lista
    
    # Dividir en dos mitades
    medio = len(lista) // 2
    izquierda = lista[:medio]
    derecha = lista[medio:]
    
    # Llamada recursiva a cada mitad
    izquierda_ordenada = merge_sort(izquierda)
    derecha_ordenada = merge_sort(derecha)
    
    # Combinar usando merge
    return merge(izquierda_ordenada, derecha_ordenada)

In [47]:
# Ejecutar Merge Sort sobre la muestra
ordenado_merge = merge_sort(datos_metric1)

# Mostrar los primeros 10 valores ordenados
print("Primeros 10 valores (Merge Sort):")
print(ordenado_merge[:10])

Primeros 10 valores (Merge Sort):
[0, 9472, 298304, 370600, 444224, 473552, 508584, 543344, 636368, 660440]


In [48]:
# Validar con función nativa de Python
ordenado_python = sorted(datos_metric1)

# Comparar ambos resultados
assert ordenado_merge == ordenado_python

print("✅ Validación exitosa: ambos resultados son iguales")

✅ Validación exitosa: ambos resultados son iguales


In [49]:
import timeit

# Tiempo de ejecución Merge Sort
tiempo_merge = timeit.timeit(
    lambda: merge_sort(datos_metric1.copy()),
    number=5
)

# Tiempo de ejecución sorted()
tiempo_python = timeit.timeit(
    lambda: sorted(datos_metric1.copy()),
    number=5
)

print("Tiempo Merge Sort:", tiempo_merge)
print("Tiempo sorted():", tiempo_python)

Tiempo Merge Sort: 0.03118780002114363
Tiempo sorted(): 0.0023607999901287258


In [50]:
import pandas as pd

resultados = pd.DataFrame({
    "Algoritmo": ["Merge Sort (Recursivo)", "sorted() (Optimizado)"],
    "Tiempo de ejecución": [tiempo_merge, tiempo_python]
})

resultados

,Algoritmo,Tiempo de ejecución
0,Merge Sort (Recursivo),0.031188
1,sorted() (Optimizado),0.002361


### Interpretación de Resultados

La recursividad es una técnica de programación en la cual una función se llama a sí misma para resolver un problema dividiéndolo en subproblemas más pequeños. En este caso, el algoritmo Merge Sort utiliza este enfoque para ordenar los datos de manera eficiente.

Merge Sort funciona bajo el paradigma de "dividir y conquistar", separando la lista en mitades sucesivas hasta llegar a listas de un solo elemento, las cuales se consideran ordenadas por definición. Luego, estas listas se combinan mediante la función merge, que construye una lista ordenada final.

Desde el punto de vista de complejidad, Merge Sort tiene una complejidad temporal de O(n log n), lo que lo hace eficiente incluso para grandes volúmenes de datos. Sin embargo, su implementación recursiva implica un mayor uso de memoria debido a la pila de llamadas.

Al comparar con la función nativa sorted(), se observa que esta última presenta un mejor rendimiento, ya que está optimizada internamente en Python.

### Conclusión

La implementación de Merge Sort permitió aplicar de manera práctica el concepto de recursividad, evidenciando cómo un problema complejo puede resolverse mediante su descomposición en partes más pequeñas. Este enfoque resulta especialmente útil desde una perspectiva formativa, ya que facilita la comprensión de algoritmos de ordenamiento eficientes.

No obstante, en un contexto aplicado al análisis de datos, se observa que el uso de funciones optimizadas como sorted() ofrece mejores tiempos de ejecución y menor sobrecarga de recursos. Por ello, si bien la recursividad es conceptualmente poderosa, su uso en entornos productivos debe evaluarse cuidadosamente, priorizando soluciones que combinen eficiencia, legibilidad y escalabilidad.